<a href="https://colab.research.google.com/github/Binamra00/rs-replication/blob/main/rel_tag_mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📖 Reader's Guide: How This Notebook Maps to the Paper

**Please read before reviewing.** This notebook is the **corpus-construction** step for the SPLASH/ISSTA 2026 SRC abstract: it mines each repository's release tags and samples the snapshots the analysis later measures. Like the analysis notebook, it predates the paper's framing, so its title and headers still use the project's original vocabulary — *"Architectural Inflection," "Goldilocks Zone," "Genesis/Terminal."* **The vocabulary is legacy; the procedure is exactly the corpus the paper describes.**

### ⚠️ Reproducibility: use the archived JSONs, not a fresh run
This notebook clones each repository at its **current `HEAD`**, and these projects keep releasing. A run today therefore reports **more** raw tags and sampled snapshots than the paper — for example, QuestDB tagged `9.4.2` on 2026-06-09, after the corpus was frozen. That growth is expected drift, **not** an inconsistency in the results.

**The frozen corpus behind Table 1 is the set of release-history JSON manifests archived on Zenodo [DOI: https://doi.org/10.5281/zenodo.20617639].** Those files — one per system — are the authoritative, immutable snapshot, and this notebook is the procedure that produced them. To reproduce the paper exactly, work from the archived JSONs. To re-mine from scratch and still match, pin each clone to the snapshot date (or filter to tags created on or before it); an unpinned re-run will not match, by design.

### Notebook → Paper correspondence

| Notebook phase | Paper element | What it produces |
|---|---|---|
| Phase 1 — Raw Tag Audit | Table 1, **Releases** column | Every release tag in the repository's history |
| Phase 2 — Clean GA Filtering | *(intermediate)* | Drops pre-releases / release candidates / milestones, leaving general-availability versions |
| Phase 3 — Era Mapping & JSON | Table 1, **Targ.** column | Groups GA versions into version eras, samples the **first and last release of each era** (genesis + terminal), and writes the per-repo JSON manifest |

Era grouping follows the paper's version-era definition: **Major.Minor eras for Commons-Lang, JUnit, Dubbo, and QuestDB; Major eras for Checkstyle** (`era_key = str(major)`), reflecting its far longer release history.

### Legacy → paper naming key

| In the notebook | In the paper |
|---|---|
| "Architectural Inflection Point," "Goldilocks Zone" | **version-era sampling** |
| "Genesis Point" / "Terminal Point" | the **first** / **last** release of each era |
| the six-archetype framing | the **five**-system corpus (a sixth candidate, Spring Framework, was excluded under the corpus's scale criterion) |

### Note on Commons-Lang tag names
The oldest Commons-Lang versions were only ever tagged as release candidates, so the sampler represents each such version by its final RC (e.g. version `3.0` ← `LANG_3_0_RC4`). The version count is unaffected; that RC commit is the version's de-facto release.

### ⛏️ Repository Tag Extraction & Validation: Apache Commons Lang
**Archetype:** Utility Library

This script establishes the longitudinal release history for **Apache Commons Lang**. Because empirical software engineering requires precise historical snapshots, we cannot rely on arbitrary commits. We must anchor our static analysis to officially recognized General Availability (GA) releases.

**The Repository Quirk: The "RC Promotion" Standard**
Unlike modern repositories that create a clean `vX.Y.Z` tag for a final release, the Apache Commons team historically utilized a Release Candidate (RC) promotion model. They would tag `RC1`, `RC2`, etc., and once a candidate passed the Apache voting process, that specific `RC` tag was published as the official GA release *without being renamed*.

To accurately map this repository's architectural evolution while ensuring absolute data integrity, this script performs a four-phase extraction:
* **Phase 1: Raw Tag Audit:** Clones the repository, normalizes legacy tag formats (e.g., converting `_` to `.`), and audits the raw tags.
* **Phase 2: Genesis & Terminal Extraction:** Resolves the highest RC tag for each official release, groups them into `Major.Minor` architectural eras, and extracts the bounding Genesis and Terminal release candidates.
* **Phase 3: Official Release Mapping:** A strict validation matrix that cross-references every extracted boundary point against the 35 verified Apache target releases to ensure zero non-official tags leaked into the dataset.
* **Phase 4: JSON Manifest Generation:** Outputs the final pipeline-ready JSON file.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/apache/commons-lang.git"
REPO_DIR = "commons-lang"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
candidates = defaultdict(list)
major_count = minor_count = patch_count = non_version_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # Normalize legacy tags (LANG_3_1 -> LANG.3.1)
    normalized_tag = tag.replace('_', '.')
    match = re.search(r'(\d+)\.(\d+)(?:\.(\d+))?', normalized_tag)

    if match:
        major = int(match.group(1))
        minor = int(match.group(2))
        patch = int(match.group(3)) if match.group(3) else 0

        if minor == 0 and patch == 0: major_count += 1
        elif patch == 0: minor_count += 1
        else: patch_count += 1
    else:
        non_version_count += 1

    # FILTER: Ignore early testing noise
    if any(x in tag.upper() for x in ['BETA', 'DEV', 'STRUTS', 'COPY', 'REPACKAGED', '_B']):
        continue

    v_match = re.search(r'(\d+\.\d+(?:\.\d+)?)', normalized_tag)
    if not v_match: continue
    base_v = v_match.group(1)

    # [REVIEWER FIX]: Prioritize true GA releases over pre-release RCs
    is_rel = tag.startswith("rel/")
    rc_match = re.search(r'RC(\d+)', tag.upper())

    # Assign a massive weight (999) to 'rel/' tags so they always win the sort.
    # Otherwise, parse the RC number, or default to 0 for un-numbered tags.
    rc_num = 999 if is_rel else (int(rc_match.group(1)) if rc_match else 0)

    candidates[base_v].append({
        'tag': tag,
        'rc': rc_num,
        'date': date[:10],
        'sha': true_sha[:12],
        'major': major,
        'minor': minor,
        'patch': patch,
        'era_key': f"{major}.{minor}"
    })

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<30} | {'Creation Date':<15} | {'SHA'}")
print("-" * 70)
for t in all_tags[-50:]:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")
print("... (truncated for brevity) ...")

print("\n" + "="*70)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print(f"Major Releases Found: {major_count}")
print(f"Minor Releases Found: {minor_count}")
print(f"Patch Releases Found: {patch_count}")
print("="*70 + "\n")

# ==========================================
# PHASE 2: GENESIS & TERMINAL EXTRACTION
# ==========================================
official_releases = [
    "3.20.0", "3.19.0", "3.18.0", "3.17.0", "3.16.0", "3.15.0", "3.14.0", "3.13.0", "3.12.0",
    "3.11", "3.10", "3.9", "3.8.1", "3.8", "3.7", "3.6", "3.5", "3.4", "3.3.2", "3.3.1", "3.3",
    "3.2.1", "3.2", "3.1", "3.0.1", "3.0", "2.6", "2.5", "2.4", "2.3", "2.2", "2.1", "2.0",
    "1.0.1", "1.0"
]

clean_ga_tags = []
for v in official_releases:
    search_versions = [v, f"{v}.0", v.replace(".0", "")]
    found_cands = []

    for sv in search_versions:
        if sv in candidates:
            found_cands.extend(candidates[sv])

    if found_cands:
        sorted_cands = sorted(found_cands, key=lambda x: x['rc'], reverse=True)
        best = sorted_cands[0]
        best['official_target'] = v # Save this for Phase 3 validation
        clean_ga_tags.append(best)

pre_release_count = len(all_tags) - non_version_count - len(clean_ga_tags)

print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 70 + "\n")

eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: [int(x) for x in k.split('.')])

for era in sorted_era_keys:
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

# ==========================================
# PHASE 3: OFFICIAL RELEASE MAPPING & TRANSPARENCY
# ==========================================
print("--- 🛡️ Phase 3: Genesis/Terminal Transparency Matrix ---")
print(f"{'Target Version':<15} | {'Extracted Tag':<25} | {'Methodological Status'}")
print("-" * 80)

verified_count = 0
discarded_count = 0
missing_count = 0

# Create quick lookup sets
target_to_clean_ga = {obj['official_target']: obj for obj in clean_ga_tags}
sampled_tag_names = {obj['tag'] for obj in sampled_tags_reversed}

for target in official_releases:
    if target in target_to_clean_ga:
        ga_obj = target_to_clean_ga[target]
        if ga_obj['tag'] in sampled_tag_names:
            # Determine if it's a Genesis or Terminal point
            era = ga_obj['era_key']
            if ga_obj['tag'] == eras[era][0]['tag']:
                status = "✅ VALID (Genesis Point)"
            else:
                status = "✅ VALID (Terminal Point)"
            verified_count += 1
            print(f"{target:<15} | {ga_obj['tag']:<25} | {status}")
        else:
            status = "⏭️ DISCARDED (Middle Patch Noise)"
            discarded_count += 1
            print(f"{target:<15} | {ga_obj['tag']:<25} | {status}")
    else:
        status = "❌ MISSING (Not Found in Repo)"
        missing_count += 1
        print(f"{target:<15} | {'[NONE]':<25} | {status}")

print("-" * 80)
print(f"Transparency Report: {verified_count} Boundaries Validated, {discarded_count} Middle Patches Discarded, {missing_count} Missing.")
print("="*80 + "\n")

# ==========================================
# PHASE 4: FINAL JSON MANIFEST
# ==========================================
print("--- 🏁 Phase 4: Final JSON Manifest for commons-lang.json ---")
print("{\n  \"versions\": [")
for i, tag_obj in enumerate(sampled_tags_reversed):
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_obj['tag']}\"{comma}")
print("  ]\n}")

Cloning https://github.com/apache/commons-lang.git...

Raw Tag Name                   | Creation Date   | SHA
----------------------------------------------------------------------
LANG_3_3_RC1                   | 2014-02-28      | af2986309044
LANG_3_4                       | 2015-04-06      | 4777c3a5e429
LANG_3_4_RC1                   | 2015-03-28      | 456836307569
LANG_3_4_RC2                   | 2015-04-03      | 980ba4455bad
LANG_3_5                       | 2016-10-20      | a9a3cef8e42f
LANG_3_5_RC1                   | 2016-10-02      | 4b2ec0798975
LANG_3_5_RC2                   | 2016-10-13      | 36f98d87b24c
LANG_3_6                       | 2017-06-12      | 09043bfa6f1f
LANG_3_6_RC1                   | 2017-04-17      | 1ec464dacd3c
LANG_3_6_RC2                   | 2017-05-17      | 3a64cf6aff40
LANG_3_6_RC3                   | 2017-06-08      | 360198dfb6a2
LANG_3_6_RC4                   | 2017-06-09      | 09043bfa6f1f
LANG_3_7                       | 2017-11-08      | 

### ⛏️ Repository Tag Extraction & Validation: Checkstyle
**Archetype:** Static Analysis / AST Tool

This script maps the evolutionary timeline of **Checkstyle**. Because Checkstyle is itself a static analysis tool designed to parse Abstract Syntax Trees (ASTs), analyzing its structural decay with a similar tool (PMD) provides a fascinating meta-analytical layer to the dataset.

**The Repository Quirk: "Minor is the new Patch" & The SVN Migration**
Checkstyle has two severe repository quirks that break standard extraction models:
1.  **The Monthly Cadence:** They follow a strict monthly release cycle, bumping the Minor version every month (e.g., 8.35, 8.36, 8.37) and reserving patches purely for emergencies. Grouping by `Major.Minor` here would incorrectly treat a single month of maintenance as a full "architectural era." We must instead group strictly by the **Major** version.
2.  **The Timestamp Corruption:** In June 2018, Checkstyle mass-migrated its legacy code from SVN to GitHub. Consequently, many v4 and v5 tags have the exact same Git creation date (`2018-06-02`), which breaks time-based chronological sorting.

To overcome these anomalies, this script executes a three-phase extraction:
* **Phase 1 & 2: Regex Filtering:** Applies an adaptive Regular Expression to capture standard and "missing third digit" tags while safely stripping out pre-releases.
* **Phase 3: Major Era Mathematical Mapping:** Groups releases strictly by their **Major** version era (v6, v7, v8...). To bypass the corrupted SVN timestamps, the script mathematically sorts the tags by their semantic version numbers (`X`, `Y`, `Z`) to flawlessly isolate the true Genesis and Terminal points of each major architecture.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/checkstyle/checkstyle.git"
REPO_DIR = "checkstyle"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (this might take a minute or two...)...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
clean_ga_tags = []

pre_release_count = 0
non_version_count = 0
major_count = minor_count = patch_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    # Store for Phase 1
    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # --- Strict Allowlist Filter Logic ---
    match = re.match(r'^(?:checkstyle-|v|release)?(\d+)[\._](\d+)(?:[\._](\d+))?$', tag)

    if not match:
        if re.search(r'\d+\.\d+(?:\.\d+)?', tag):
            pre_release_count += 1
        else:
            non_version_count += 1
        continue

    major = int(match.group(1))
    minor = int(match.group(2))
    # If the tag is just "checkstyle-8.1", treat the patch as 0 mathematically
    patch = int(match.group(3)) if match.group(3) else 0

    clean_ga_tags.append({
        'tag': tag,
        'date': date[:10],
        'sha': true_sha[:12],
        'era_key': str(major), # Grouping purely by MAJOR era
        'major': major,
        'minor': minor,
        'patch': patch
    })

    # SemVer Metrics
    if minor == 0 and patch == 0: major_count += 1
    elif patch == 0: minor_count += 1
    else: patch_count += 1

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<30} | {'Creation Date':<15} | {'SHA'}")
print("-" * 70)

# Iterate over the entire all_tags list instead of just the last 50
for t in all_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print("="*70 + "\n")

# ==========================================
# PHASE 2: CLEAN GA EXTRACTION
# ==========================================
print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 40)
print(f"  ├─ Major Releases (X.0.0):  {major_count}")
print(f"  ├─ Minor Releases (X.Y.0):  {minor_count}")
print(f"  └─ Patch Releases (X.Y.Z):  {patch_count}")
print("="*70 + "\n")

# ==========================================
# PHASE 3: EXTRACT INFLECTION THRESHOLDS
# ==========================================
eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: int(k))

for era in sorted_era_keys:
    # Sorting mathematically by version numbers to bypass the 2018 GitHub mass-migration date glitch
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

# --- Visual Verification Table ---
print("--- 🗺️ Phase 3: Architectural Boundary Map (Major Eras) ---")
print(f"{'Sampled Tag':<25} | {'Era Role':<20} | {'Release Date'}")
print("-" * 65)
for tag_obj in sampled_tags_reversed:
    tag_name = tag_obj['tag']
    # If it is the absolute lowest minor/patch in this Major era, it's the Genesis
    role = "Genesis Point" if tag_obj == eras[str(tag_obj['major'])][0] else "Terminal Point"
    print(f"{tag_name:<25} | {role:<20} | {tag_obj['date']}")

# --- Final Clean JSON Output ---
print("\n--- 🏁 Final JSON Manifest for checkstyle.json ---")
print("{\n  \"versions\": [")
for i, tag_obj in enumerate(sampled_tags_reversed):
    tag_name = tag_obj['tag']
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_name}\"{comma}")
print("  ]\n}")

Cloning https://github.com/checkstyle/checkstyle.git (this might take a minute or two...)...

Raw Tag Name                   | Creation Date   | SHA
----------------------------------------------------------------------
bcel                           | 2002-09-08      | 01d0f17ca742
checkstyle-4.4                 | 2018-06-02      | c77953b8a8ac
checkstyle-5.2                 | 2018-06-02      | c77953b8a8ac
checkstyle-5.3                 | 2018-06-02      | c77953b8a8ac
checkstyle-5.4                 | 2018-06-02      | c77953b8a8ac
checkstyle-5.5                 | 2018-06-02      | c77953b8a8ac
checkstyle-5.6                 | 2018-06-02      | c77953b8a8ac
checkstyle-5.7                 | 2018-06-02      | c77953b8a8ac
checkstyle-5.8                 | 2014-10-04      | 8e4c7f743e5e
checkstyle-5.9                 | 2014-10-16      | 068b6d4a87d5
checkstyle-6.0                 | 2014-10-23      | a780f92fe771
checkstyle-6.1                 | 2014-11-13      | bc5b24ddfd19
checkstyle-6

### ⛏️ Repository Tag Extraction & Validation: Apache Dubbo
**Archetype:** Microservices / RPC Framework

This script maps the architectural evolution of **Apache Dubbo**, a high-performance, Java-based open-source RPC framework. Because Dubbo handles complex network routing, service discovery, and load balancing, its architectural inflection points provide unique insights into the technical debt profile of heavily distributed systems.

**The Repository Quirk: The `dubbo-` Prefix & Incubation History**
Apache Dubbo has a unique development history, having transitioned from an internal Alibaba project to a top-level Apache Software Foundation project. Consequently, its tagging culture is mixed. Formal releases traditionally use a `dubbo-X.Y.Z` prefix, but earlier or transitional tags sometimes use `vX.Y.Z` or just pure `X.Y.Z`. Additionally, the repository is filled with `.preview` and `-beta` tags that must be rigorously filtered out.

To capture the true structural evolution of the framework, this script utilizes a three-phase extraction:
* **Phase 1 & 2: Strict Regex Allowlist:** Clones the repository and applies a targeted Regular Expression (`^(?:dubbo-|v)?(\d+)\.(\d+)\.(\d+)$`) to safely capture all valid GA formats while completely eradicating previews, betas, and release candidates.
* **Phase 3: Architectural Boundary Mapping:** Groups the surviving, clean GA releases into their `Major.Minor` architectural eras (e.g., `2.6`, `2.7`, `3.0`). It then extracts the chronologically first "Genesis Point" and the final "Terminal Point" of each era. This isolates the exact boundaries where major structural shifts and feature additions occurred, stripping away the noise of over 100 minor bug-fix patches.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/apache/dubbo.git"
REPO_DIR = "dubbo"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (this might take a minute or two...)...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
clean_ga_tags = []

pre_release_count = 0
non_version_count = 0
major_count = minor_count = patch_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    # Store for Phase 1
    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # --- Strict Allowlist Filter Logic ---
    match = re.match(r'^(?:dubbo-|v)?(\d+)\.(\d+)\.(\d+)$', tag)

    if not match:
        if re.search(r'\d+\.\d+\.\d+', tag):
            pre_release_count += 1
        else:
            non_version_count += 1
        continue

    major, minor, patch = int(match.group(1)), int(match.group(2)), int(match.group(3))

    clean_ga_tags.append({
        'tag': tag,
        'date': date[:10],
        'sha': true_sha[:12],
        'era_key': f"{major}.{minor}", # Group by Major.Minor era
        'major': major,                # [FIX]: Save for mathematical sorting
        'minor': minor,
        'patch': patch
    })

    # SemVer Metrics
    if minor == 0 and patch == 0: major_count += 1
    elif patch == 0: minor_count += 1
    else: patch_count += 1

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<30} | {'Creation Date':<15} | {'SHA'}")
print("-" * 70)
for t in all_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print("="*70 + "\n")

# ==========================================
# PHASE 2: CLEAN GA EXTRACTION
# ==========================================
print(f"{'Clean GA Release':<30} | {'Release Date':<15} | {'SHA'}")
print("-" * 70)
for t in clean_ga_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 40)
print(f"  ├─ Major Releases (X.0.0):  {major_count}")
print(f"  ├─ Minor Releases (X.Y.0):  {minor_count}")
print(f"  └─ Patch Releases (X.Y.Z):  {patch_count}")
print("="*70 + "\n")

# ==========================================
# PHASE 3: EXTRACT INFLECTION THRESHOLDS
# ==========================================
eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: [int(x) for x in k.split('.')])

for era in sorted_era_keys:
    # [FIX]: Sort mathematically by SemVer, NOT by Git date
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

# --- Visual Verification Table ---
print("--- 🗺️ Phase 3: Architectural Boundary Map ---")
print(f"{'Sampled Tag':<25} | {'Era Role':<20} | {'Release Date'}")
print("-" * 65)
for tag_obj in sampled_tags_reversed:
    tag_name = tag_obj['tag']
    # [FIX]: Determine Genesis by its position in the era list, not just if it ends in .0
    role = "Genesis Point" if tag_obj == eras[tag_obj['era_key']][0] else "Terminal Point"
    print(f"{tag_name:<25} | {role:<20} | {tag_obj['date']}")
print("\n")

# --- Final Clean JSON Output ---
print("--- 🏁 Phase 3: Final JSON Manifest for dubbo.json ---")
print("{\n  \"versions\": [")

for i, tag_obj in enumerate(sampled_tags_reversed):
    tag_name = tag_obj['tag']
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_name}\"{comma}")

print("  ]\n}")

Cloning https://github.com/apache/dubbo.git (this might take a minute or two...)...

Raw Tag Name                   | Creation Date   | SHA
----------------------------------------------------------------------
2.7.6                          | 2020-03-23      | af4cff2846bd
3.0.0.preview                  | 2021-03-19      | 95b776385ee8
3.2.12                         | 2024-03-12      | 1dd6a4aaa881
dubbo-2.0.7                    | 2011-10-27      | ad3ec761d2fc
dubbo-2.0.8                    | 2011-12-27      | de779460b5b4
dubbo-2.0.9                    | 2011-12-15      | dd677eae2cac
dubbo-2.0.10                   | 2012-01-06      | 827433d4cdbe
dubbo-2.0.11                   | 2012-01-13      | b48a27f4422e
dubbo-2.0.12                   | 2012-02-06      | 271ea1231b7e
dubbo-2.0.13                   | 2012-02-29      | dd183a8be6f8
dubbo-2.0.14                   | 2012-03-02      | 73253aac4b87
dubbo-2.1.0                    | 2012-03-14      | 6e18491c649e
dubbo-2.1.1          

### ⛏️ Repository Tag Extraction & Validation: JUnit 5
**Archetype:** Testing Framework

This script maps the evolutionary timeline of **JUnit 5**, the industry-standard testing framework for the Java ecosystem.

**The Repository Quirk: The 'r' Prefix & SemVer Milestones**
Unlike Apache Commons, the JUnit team adheres much closer to modern Semantic Versioning (SemVer) but employs a specific prefix: all official JUnit 5 (and the upcoming JUnit 6) tags begin with `r` (e.g., `r5.8.2`). Furthermore, their development lifecycle relies heavily on public milestones and release candidates tagged with hyphens (e.g., `r5.0.0-M1`, `r5.0.0-RC2`).

To isolate the true architectural inflection points from the preliminary testing noise, this script executes a structured three-phase extraction:
* **Phase 1: Raw Tag Audit:** Clones the repository, extracts all tags, and parses them to calculate the exact distribution of Major, Minor, and Patch releases across the project's history.
* **Phase 2: Genesis & Terminal Extraction (Clean GA):** Applies a strict filter to isolate only `r5.*` and `r6.*` tags while programmatically stripping out any tag containing a hyphen (`-`). It then groups these pure General Availability (GA) releases into their `Major.Minor` eras to extract the bounding Genesis and Terminal points.
* **Phase 3: JSON Manifest Generation:** Outputs the final pipeline-ready JSON file.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/junit-team/junit-framework.git"
REPO_DIR = "junit5"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (this might take a minute or two...)...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]

result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
clean_ga_tags = []

pre_release_count = 0
non_version_count = 0
major_count = minor_count = patch_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # Analyze SemVer for Audit Metrics
    match = re.search(r'(\d+)\.(\d+)\.(\d+)', tag)
    if match:
        maj, min, pat = match.groups()
        if min == '0' and pat == '0': major_count += 1
        elif pat == '0': minor_count += 1
        else: patch_count += 1

    # --- Strict Allowlist Filter Logic ---
    # FILTER 1: Only look at JUnit 5 and the new JUnit 6 tags
    if not (tag.startswith('r5.') or tag.startswith('r6.')):
        non_version_count += 1
        continue

    # FILTER 2: Exclude all pre-releases (Milestones, RCs, Alphas)
    if '-' in tag:
        pre_release_count += 1
        continue

    # If it survives, parse it into an era object
    clean_match = re.match(r'^r(\d+)\.(\d+)\.(\d+)$', tag)
    if clean_match:
        major = int(clean_match.group(1))
        minor = int(clean_match.group(2))
        patch = int(clean_match.group(3))

        clean_ga_tags.append({
            'tag': tag,
            'date': date[:10],
            'sha': true_sha[:12],
            'era_key': f"{major}.{minor}",
            'major': major,
            'minor': minor,
            'patch': patch
        })

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<25} | {'Release Date':<15} | {'SHA'}")
print("-" * 65)
for t in all_tags: # Truncated to 50 for Colab readability
    print(f"{t['tag']:<25} | {t['date']:<15} | {t['sha']}")


print("\n" + "="*65)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print(f"Major Releases Found: {major_count}")
print(f"Minor Releases Found: {minor_count}")
print(f"Patch Releases Found: {patch_count}")
print("="*65 + "\n")


# ==========================================
# PHASE 2: GENESIS & TERMINAL EXTRACTION
# ==========================================
eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: [int(x) for x in k.split('.')])

for era in sorted_era_keys:
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 65)

print("\n--- 🗺️ Phase 2: Architectural Boundary Map ---")
print(f"{'Sampled Tag':<25} | {'Era Role':<20} | {'Release Date'}")
print("-" * 65)
for tag_obj in sampled_tags_reversed:
    role = "Genesis Point (.0)" if tag_obj == eras[tag_obj['era_key']][0] else "Terminal Point"
    print(f"{tag_obj['tag']:<25} | {role:<20} | {tag_obj['date']}")
print("\n")


# ==========================================
# PHASE 3: FINAL JSON MANIFEST
# ==========================================
print("--- 🏁 Phase 3: Final JSON Manifest for junit5.json ---")
print("{\n  \"versions\": [")
for i, tag_obj in enumerate(sampled_tags_reversed):
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_obj['tag']}\"{comma}")
print("  ]\n}")

Cloning https://github.com/junit-team/junit-framework.git (this might take a minute or two...)...

Raw Tag Name              | Release Date    | SHA
-----------------------------------------------------------------
prototype-0               | 2015-10-27      | ccdbc6d9a3a0
prototype-1               | 2015-12-03      | 80fd12170d4a
r5.0.0                    | 2017-09-10      | 4bf35e5d84fc
r5.0.0-ALPHA              | 2016-02-01      | f7ba2ef3bd97
r5.0.0-M1                 | 2016-07-07      | d223d4fbbc89
r5.0.0-M2                 | 2016-07-23      | 178385e5a87b
r5.0.0-M3                 | 2016-11-30      | 8105df0b3d82
r5.0.0-M4                 | 2017-04-01      | 481aa778751f
r5.0.0-M5                 | 2017-07-04      | 3a209fdfb0c9
r5.0.0-M6                 | 2017-07-18      | 3e6482ab8b0d
r5.0.0-RC1                | 2017-07-30      | 2a8ebf0413b6
r5.0.0-RC2                | 2017-07-30      | 1636582d3d13
r5.0.0-RC3                | 2017-08-23      | e0af182b1440
r5.0.1            

### ⛏️ Repository Tag Extraction & Validation: QuestDB
**Archetype:** High-Performance Time-Series Database

This script maps the evolutionary timeline of **QuestDB**. As a high-performance database optimized for zero-GC (garbage collection) Java and raw hardware efficiency, its technical debt profile is highly sensitive to core storage engine rewrites.

**The Repository Quirk: The Optional Patch Digit**
QuestDB maintains a very clean tagging culture, but with one mathematical quirk: developers frequently omit the third `.0` digit when tagging the start of a new Major or Minor era (e.g., tagging `1.0` instead of `1.0.0`, or `6.1` instead of `6.1.0`). A standard Semantic Versioning parser would discard these as invalid, accidentally deleting the true Genesis points of the architecture.

To capture these shifts precisely, this script utilizes a three-phase extraction:
* **Phase 1 & 2: Adaptive Regex Filtering:** Uses an advanced Regular Expression (`^(?:v)?(\d+)\.(\d+)(?:\.(\d+))?$`) that makes the third digit optional. It mathematically infers the missing `.0` to ensure true initial releases are captured while still safely discarding alphas, betas, and release candidates.
* **Phase 3: Inflection Threshold Mapping:** Groups the clean GA releases into their `Major.Minor` architectural eras. It extracts the "Genesis Point" (the inferred `.0` release introducing the new architecture) and the "Terminal Point" (the final patch representing the mature architecture) of each era, isolating the bounds of structural code decay.

In [ ]:
import subprocess
import re
import os
from collections import defaultdict

REPO_URL = "https://github.com/questdb/questdb.git"
REPO_DIR = "questdb"

# 1. Clone the repository ONLY if it doesn't already exist
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (this might take a minute or two...)...\n")
    subprocess.run(["git", "clone", "--bare", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at '{REPO_DIR}'. Skipping clone...\n")

# 2. Extract Tags
cmd = [
    "git", "for-each-ref",
    "--sort=version:refname",
    "--format=%(refname:short)|%(creatordate:iso8601)|%(*objectname)|%(objectname)",
    "refs/tags"
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, check=True)
lines = result.stdout.strip().split("\n")

all_tags = []
clean_ga_tags = []

pre_release_count = 0
non_version_count = 0
major_count = minor_count = patch_count = 0

for line in lines:
    if not line.strip(): continue
    tag, date, peeled_sha, tag_sha = line.split('|')
    true_sha = peeled_sha if peeled_sha else tag_sha

    # Store for Phase 1
    all_tags.append({'tag': tag, 'date': date[:10], 'sha': true_sha[:12]})

    # --- UPDATED ALLOWLIST LOGIC ---
    # Now allows X.Y.Z OR X.Y (makes the third digit optional)
    match = re.match(r'^(?:v)?(\d+)\.(\d+)(?:\.(\d+))?$', tag)

    if not match:
        if re.search(r'\d+\.\d+', tag):
            pre_release_count += 1
        else:
            non_version_count += 1
        continue

    major = int(match.group(1))
    minor = int(match.group(2))
    # If the tag is just "1.0", treat the patch as 0
    patch = int(match.group(3)) if match.group(3) else 0

    clean_ga_tags.append({
        'tag': tag,
        'date': date[:10],
        'sha': true_sha[:12],
        'era_key': f"{major}.{minor}",
        'major': major,                # [FIX]: Required for mathematical sorting
        'minor': minor,
        'patch': patch
    })

    # SemVer Metrics
    if minor == 0 and patch == 0: major_count += 1
    elif patch == 0: minor_count += 1
    else: patch_count += 1

# ==========================================
# PHASE 1: FULL AUDIT OUTPUT
# ==========================================
print(f"{'Raw Tag Name':<30} | {'Creation Date':<15} | {'SHA'}")
print("-" * 70)
for t in all_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 📊 Phase 1: Raw Audit Summary ---")
print(f"Total Raw Tags Mined: {len(all_tags)}")
print("="*70 + "\n")

# ==========================================
# PHASE 2: CLEAN GA EXTRACTION
# ==========================================
print(f"{'Clean GA Release':<30} | {'Release Date':<15} | {'SHA'}")
print("-" * 70)
for t in clean_ga_tags:
    print(f"{t['tag']:<30} | {t['date']:<15} | {t['sha']}")

print("\n" + "="*70)
print("--- 🏁 Phase 2: Final Filter Summary ---")
print(f"Total Raw Tags:               {len(all_tags)}")
print(f"[-] Removed Non-Versions:     {non_version_count}")
print(f"[-] Removed Pre-Releases:     {pre_release_count}")
print(f"[+] Total Clean GA Releases:  {len(clean_ga_tags)}")
print("-" * 40)
print(f"  ├─ Major Releases (X.0.0):  {major_count}")
print(f"  ├─ Minor Releases (X.Y.0):  {minor_count}")
print(f"  └─ Patch Releases (X.Y.Z):  {patch_count}")
print("="*70 + "\n")

# ==========================================
# PHASE 3: EXTRACT INFLECTION THRESHOLDS
# ==========================================
eras = defaultdict(list)
for tag_obj in clean_ga_tags:
    eras[tag_obj['era_key']].append(tag_obj)

sampled_tags = []
sorted_era_keys = sorted(eras.keys(), key=lambda k: [int(x) for x in k.split('.')])

for era in sorted_era_keys:
    # [FIX]: Sort mathematically by SemVer, NOT by Git date to prevent backport time-travel
    era_tags = sorted(eras[era], key=lambda x: (x['major'], x['minor'], x['patch']))

    genesis = era_tags[0]
    sampled_tags.append(genesis)

    terminal = era_tags[-1]
    if terminal['tag'] != genesis['tag']:
        sampled_tags.append(terminal)

sampled_tags_reversed = list(reversed(sampled_tags))

print("--- 🗺️ Phase 3: Architectural Boundary Map ---")
print(f"{'Sampled Tag':<25} | {'Era Role':<20} | {'Release Date'}")
print("-" * 65)
for tag_obj in sampled_tags_reversed:
    tag_name = tag_obj['tag']
    # [FIX]: Dynamically determine Genesis by array position, not hardcoded patch numbers
    role = "Genesis Point" if tag_obj == eras[tag_obj['era_key']][0] else "Terminal Point"
    print(f"{tag_name:<25} | {role:<20} | {tag_obj['date']}")

print("\n--- 🏁 Final JSON Manifest for questdb.json ---")
print("{\n  \"versions\": [")
for i, tag_obj in enumerate(sampled_tags_reversed):
    tag_name = tag_obj['tag']
    comma = "," if i < len(sampled_tags_reversed) - 1 else ""
    print(f"    \"{tag_name}\"{comma}")
print("  ]\n}")

Cloning https://github.com/questdb/questdb.git (this might take a minute or two...)...

Raw Tag Name                   | Creation Date   | SHA
----------------------------------------------------------------------
1.0                            | 2014-05-06      | 20adbb1c95ee
1.0.1                          | 2014-05-12      | e13a713c962a
1.0.2                          | 2014-06-01      | fe150e6c80b2
1.0.3                          | 2014-06-18      | 761039f1fdc8
2.0.0                          | 2014-08-18      | 01621cde95de
2.0.1                          | 2014-08-31      | 50508cccb7fd
2.1.0                          | 2014-10-13      | e5dbc5f4f3cc
3.0.0                          | 2018-10-07      | 5a6043601bd5
4.0.0                          | 2019-11-19      | f335c014d946
4.0.1                          | 2019-11-25      | dfd5bff88c4f
4.0.2                          | 2019-12-04      | 604fab9ee696
4.0.3                          | 2019-12-09      | d0b7c64bb3a5
4.0.4             